In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
from plot_utils import topk_barplot, cumsum_plot

In [ ]:
sample_size = 25

In [ ]:
current_dir = os.getcwd()

data_dir = "tech_challenge-data-us_flights_ml"

In [ ]:
flights_csv = "flights.csv"
df_flights = pd.read_csv(os.path.join(current_dir, data_dir, flights_csv))

In [ ]:
df_flights.info()

In [ ]:
df_flights_considered = (
    df_flights[(df_flights["CANCELLED"] == 0) & (df_flights["DIVERTED"] == 0)]
    .drop(columns=["CANCELLED", "CANCELLATION_REASON", "DIVERTED"])
)

In [ ]:
df_flights_considered.info()

In [ ]:
df_flights_considered.sample(n=sample_size)

In [ ]:
wd_mapping = {
    1: "MON",
    2: "TUE",
    3: "WED",
    4: "THU",
    5: "FRI",
    6: "SAT",
    7: "SUN",
}

df_flights_considered["DAY_OF_WEEK_ABBR"] = pd.Categorical(
    df_flights_considered["DAY_OF_WEEK"],
    categories=wd_mapping.keys(),
    ordered=True
).rename_categories(wd_mapping)

In [ ]:
df_flights_considered["ROUTE"] = df_flights_considered["ORIGIN_AIRPORT"].astype(str) + "-" + df_flights_considered["DESTINATION_AIRPORT"].astype(str)

In [ ]:
df_flights_considered[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL", "SCHEDULED_TIME", "DEPARTURE_TIME", "DEPARTURE_DELAY", "ARRIVAL_TIME", "ELAPSED_TIME", "ARRIVAL_DELAY"]].isna().sum()

In [ ]:
df_flights_considered["HOUR"] = df_flights_considered["SCHEDULED_DEPARTURE"] // 100
df_flights_considered["MINUTE"] = df_flights_considered["SCHEDULED_DEPARTURE"] % 100
df_flights_considered["DT_SCHEDULED_DEPARTURE"] = pd.to_datetime(df_flights_considered[["YEAR", "MONTH", "DAY", "HOUR", "MINUTE"]])

df_flights_considered["TD_SCHEDULED_TIME"] = pd.to_timedelta(df_flights_considered["SCHEDULED_TIME"], unit="min")
df_flights_considered["TD_DEPARTURE_DELAY"] = pd.to_timedelta(df_flights_considered["DEPARTURE_DELAY"], unit="min")
df_flights_considered["TD_ELAPSED_TIME"] = pd.to_timedelta(df_flights_considered["ELAPSED_TIME"], unit="min")

df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"] = df_flights_considered["DT_SCHEDULED_DEPARTURE"] + df_flights_considered["TD_SCHEDULED_TIME"]
df_flights_considered["DT_ARRIVAL_CALC"] = df_flights_considered["DT_SCHEDULED_DEPARTURE"] + df_flights_considered["TD_DEPARTURE_DELAY"] + df_flights_considered["TD_ELAPSED_TIME"]

df_flights_considered["INT_ARRIVAL_DELAY_CALC_1"] = (df_flights_considered["DT_ARRIVAL_CALC"] - df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"]).dt.total_seconds() / 60
df_flights_considered["INT_ARRIVAL_DELAY_CALC_2"] = (df_flights_considered["TD_DEPARTURE_DELAY"] + df_flights_considered["TD_ELAPSED_TIME"] - df_flights_considered["TD_SCHEDULED_TIME"]).dt.total_seconds() / 60

## Sanity checks

In [ ]:
df_flights_considered[["DT_SCHEDULED_DEPARTURE", "TD_SCHEDULED_TIME", "DT_SCHEDULED_ARRIVAL_CALC", "SCHEDULED_ARRIVAL"]].sample(n=sample_size)

In [ ]:
(df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"].dt.minute != (df_flights_considered["SCHEDULED_ARRIVAL"] % 100)).sum()

In [ ]:
df_flights_considered.loc[
    df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"].dt.minute != (df_flights_considered["SCHEDULED_ARRIVAL"] % 100),
    ["AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DT_SCHEDULED_DEPARTURE", "TD_SCHEDULED_TIME", "DT_SCHEDULED_ARRIVAL_CALC", "SCHEDULED_ARRIVAL"]
]

In [ ]:
(df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"].dt.hour - (df_flights_considered["SCHEDULED_ARRIVAL"] // 100)).value_counts()

In [ ]:
df_flights_considered[["DT_SCHEDULED_DEPARTURE", "TD_DEPARTURE_DELAY", "DEPARTURE_TIME", "TD_ELAPSED_TIME", "DT_ARRIVAL_CALC", "ARRIVAL_TIME"]].sample(n=sample_size)

In [ ]:
(df_flights_considered["DT_ARRIVAL_CALC"].dt.minute != (df_flights_considered["ARRIVAL_TIME"] % 100)).sum()

In [ ]:
df_flights_considered.loc[
    df_flights_considered["DT_ARRIVAL_CALC"].dt.minute != (df_flights_considered["ARRIVAL_TIME"] % 100),
    ["AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DT_SCHEDULED_DEPARTURE", "TD_DEPARTURE_DELAY", "DEPARTURE_TIME", "TD_ELAPSED_TIME", "DT_ARRIVAL_CALC", "ARRIVAL_TIME"]
]

In [ ]:
(df_flights_considered["DT_ARRIVAL_CALC"].dt.hour - (df_flights_considered["ARRIVAL_TIME"] // 100)).value_counts()

In [ ]:
df_flights_considered[["ARRIVAL_TIME", "SCHEDULED_ARRIVAL", "ARRIVAL_DELAY", "INT_ARRIVAL_DELAY_CALC_1", "INT_ARRIVAL_DELAY_CALC_2"]].sample(n=sample_size)

In [ ]:
delay_cols = ["ARRIVAL_DELAY", "INT_ARRIVAL_DELAY_CALC_1", "INT_ARRIVAL_DELAY_CALC_2"]
df_flights_considered[delay_cols] = df_flights_considered[delay_cols].astype(int)

In [ ]:
(df_flights_considered["INT_ARRIVAL_DELAY_CALC_1"] != df_flights_considered["ARRIVAL_DELAY"]).sum()

In [ ]:
df_flights_considered.loc[
    df_flights_considered["INT_ARRIVAL_DELAY_CALC_1"] != df_flights_considered["ARRIVAL_DELAY"],
    ["AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DT_ARRIVAL_CALC", "ARRIVAL_TIME", "SCHEDULED_ARRIVAL", "ARRIVAL_DELAY", "INT_ARRIVAL_DELAY_CALC_1"]
]

In [ ]:
(df_flights_considered["INT_ARRIVAL_DELAY_CALC_2"] != df_flights_considered["ARRIVAL_DELAY"]).sum()

In [ ]:
df_flights_considered.loc[
    df_flights_considered["INT_ARRIVAL_DELAY_CALC_2"] != df_flights_considered["ARRIVAL_DELAY"],
    ["AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DT_ARRIVAL_CALC", "ARRIVAL_TIME", "SCHEDULED_ARRIVAL", "ARRIVAL_DELAY", "INT_ARRIVAL_DELAY_CALC_2"]
]

## Delayed

In [ ]:
tol_min = 0
df_flights_considered["DELAYED"] = df_flights_considered["ARRIVAL_DELAY"] > tol_min

In [ ]:
df_flights_considered["DELAYED"].value_counts(normalize=True)

In [ ]:
delay_reason_cols = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]
df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].isna().sum()

In [ ]:
df_flights_considered.loc[df_flights_considered["DELAYED"], ["ARRIVAL_DELAY"] + delay_reason_cols].sample(n=sample_size)

In [ ]:
df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].sum(axis=1, skipna=False).value_counts().sort_index()

In [ ]:
tol_min = 15
df_flights_considered["DELAYED"] = df_flights_considered["ARRIVAL_DELAY"] >= tol_min

In [ ]:
df_flights_considered["DELAYED"].value_counts(normalize=True)

In [ ]:
df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].isna().sum()

In [ ]:
df_flights_considered.loc[~df_flights_considered["DELAYED"], delay_reason_cols].isna().sum()

In [ ]:
df_flights_considered[~df_flights_considered["DELAYED"]].shape[0]

In [ ]:
df_flights_delayed = df_flights_considered[df_flights_considered["DELAYED"]]

In [ ]:
df_flights_delayed.info()

In [ ]:
df_flights_delayed.sample(n=sample_size)

In [ ]:
df_flights_delayed["MONTH"].value_counts(normalize=True)

In [ ]:
df_flights_delayed["MONTH"].value_counts().sort_index().plot(kind="bar", figsize=(8, 5))

In [ ]:
df_flights_delayed["DAY"].value_counts(normalize=True)

In [ ]:
df_flights_delayed["DAY"].value_counts().sort_index().plot(kind="bar", figsize=(8, 5))

In [ ]:
df_flights_delayed["DAY_OF_WEEK_ABBR"].value_counts(normalize=True)

In [ ]:
df_flights_delayed["DAY_OF_WEEK_ABBR"].value_counts().sort_index().plot(kind="bar", figsize=(8, 5))

In [ ]:
df_flights_delayed["HOUR"].value_counts(normalize=True)

In [ ]:
df_flights_delayed["HOUR"].value_counts().sort_index().plot(kind="bar", figsize=(8, 5))

In [ ]:
df_flights_delayed["ORIGIN_AIRPORT"].value_counts(normalize=True)

In [ ]:
_ = topk_barplot(df_flights_delayed["ORIGIN_AIRPORT"], normalize=True, k=25, group_others=False)
plt.show()

In [ ]:
_ = cumsum_plot(df_flights_delayed["ORIGIN_AIRPORT"], annotate=True, threshold=0.8)
plt.show()

In [ ]:
df_flights_delayed["DESTINATION_AIRPORT"].value_counts(normalize=True)

In [ ]:
_ = topk_barplot(df_flights_delayed["DESTINATION_AIRPORT"], normalize=True, k=25, group_others=False)
plt.show()

In [ ]:
_ = cumsum_plot(df_flights_delayed["DESTINATION_AIRPORT"], annotate=True, threshold=0.8)
plt.show()

In [ ]:
df_flights_delayed["ROUTE"].value_counts(normalize=True)

In [ ]:
_ = topk_barplot(df_flights_delayed["ROUTE"], normalize=True, k=25, group_others=False)
plt.show()

In [ ]:
_ = cumsum_plot(df_flights_delayed["ROUTE"], annotate=True, threshold=0.8)
plt.show()

In [ ]:
df_flights_delayed["AIRLINE"].value_counts(normalize=True)

In [ ]:
df_flights_delayed["AIRLINE"].value_counts().sort_index().plot(kind="bar", figsize=(8, 5))

In [ ]:
(df_flights_delayed["MONTH"].value_counts() / df_flights_considered["MONTH"].value_counts()).sort_values(ascending=False)

In [ ]:
(df_flights_delayed["DAY"].value_counts() / df_flights_considered["DAY"].value_counts()).sort_values(ascending=False)

In [ ]:
(df_flights_delayed["DAY_OF_WEEK_ABBR"].value_counts() / df_flights_considered["DAY_OF_WEEK_ABBR"].value_counts()).sort_values(ascending=False)

In [ ]:
(df_flights_delayed["HOUR"].value_counts() / df_flights_considered["HOUR"].value_counts()).sort_values(ascending=False)

In [ ]:
(df_flights_delayed["ORIGIN_AIRPORT"].value_counts() / df_flights_considered["ORIGIN_AIRPORT"].value_counts()).sort_values(ascending=False)

In [ ]:
(df_flights_delayed["DESTINATION_AIRPORT"].value_counts() / df_flights_considered["DESTINATION_AIRPORT"].value_counts()).sort_values(ascending=False)

In [ ]:
(df_flights_delayed["ROUTE"].value_counts() / df_flights_considered["ROUTE"].value_counts()).sort_values(ascending=False)

In [ ]:
(df_flights_delayed["AIRLINE"].value_counts() / df_flights_considered["AIRLINE"].value_counts()).sort_values(ascending=False)